# RASA : Mission Case Study
Run ranking and basket optimization directly from artifacts without a server or retraining.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from rasa.data.schema import load_processed_dataset
from rasa.missions.validation import Mission, Preferences, validate_mission
from rasa.optimization.basket import optimize_basket
from rasa.services.catalog import CatalogService
from rasa.services.model_registry import ModelRegistry

In [2]:
bundle = load_processed_dataset()
catalog = CatalogService().load_from_bundle(bundle).catalog
registry = ModelRegistry().load()
assert registry.ready, registry.error
mission = Mission(
    goal='Build a minimalist everyday campus kit',
    required_categories=['bag', 'stationery', 'drinkware'],
    optional_categories=['organizer'], max_budget=250_000, max_merchants=2,
    preferences=Preferences(style=['minimalist'], colors=['black', 'gray'], minimum_rating=4.2),
)
validation = validate_mission(mission, catalog, bundle.categories)
validation.to_dict()

D:\Project\production-recommendation-system\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.4.1.post1 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
D:\Project\production-recommendation-system\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.4.1.post1 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
D:\Project\production-recommendation-system\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimat

{'valid': True,
 'errors': [],
 'warnings': [{'code': 'merchant_conflict',
   'message': 'Required categories are estimated to need at least 3 merchants, but the current limit is 2.',
   'field': 'max_merchants',
   'severity': 'warning'}],
 'normalized': {'goal': 'Build a minimalist everyday campus kit',
  'required_categories': ['bag', 'stationery', 'drinkware'],
  'optional_categories': ['organizer'],
  'max_budget': 250000.0,
  'max_merchants': 2,
  'preferences': {'style': ['minimalist'],
   'colors': ['black', 'gray'],
   'brands': [],
   'minimum_rating': 4.2,
   'max_item_price': None}}}

In [3]:
ranked = registry.ranker.rank(catalog, validation.normalized, strategy='hybrid_mission', top_k=10)
basket = optimize_basket(ranked.scored, validation.normalized, catalog, ranked.diagnostics.get('candidate_generation', {}))
basket.to_dict()

{'feasible': True,
 'status': 'feasible',
 'items': [{'product_id': 'p0084',
   'product_name': 'Lentera Highlighter 24',
   'category_id': 'stationery',
   'category_name': 'Stationery',
   'price': 32500.0,
   'rating': 4.67,
   'merchant_id': 'm026',
   'merchant_name': 'Compact Warehouse',
   'score': 0.743471,
   'role': 'required',
   'reasons': []},
  {'product_id': 'p0221',
   'product_name': 'Aksara Desk Organizer 41',
   'category_id': 'organizer',
   'category_name': 'Organizer',
   'price': 25500.0,
   'rating': 4.74,
   'merchant_id': 'm009',
   'merchant_name': 'Karsa Technology',
   'score': 0.604079,
   'role': 'optional',
   'reasons': []},
  {'product_id': 'p0173',
   'product_name': 'Sagara Bottle 53',
   'category_id': 'drinkware',
   'category_name': 'Drinkware',
   'price': 30000.0,
   'rating': 4.88,
   'merchant_id': 'm009',
   'merchant_name': 'Karsa Technology',
   'score': 0.468984,
   'role': 'required',
   'reasons': []},
  {'product_id': 'p0038',
   'produ

In [4]:
assert basket.total_price <= mission.max_budget
assert basket.merchant_count <= mission.max_merchants
assert basket.required_category_coverage == 1.0
[(item.product_name, item.category_name, item.price, item.merchant_name) for item in basket.items]

[('Lentera Highlighter 24', 'Stationery', 32500.0, 'Compact Warehouse'),
 ('Aksara Desk Organizer 41', 'Organizer', 25500.0, 'Karsa Technology'),
 ('Sagara Bottle 53', 'Drinkware', 30000.0, 'Karsa Technology'),
 ('Nusakita Tote Bag 38', 'Bags', 73000.0, 'Karsa Technology')]